# Day 2 — NLP Foundational Concepts
## 30 Days of AI: From NLP to LLMs

---

Welcome to Day 2. Yesterday we covered what NLP is and why it matters. Today we go hands-on with the building blocks that every NLP pipeline is built on.

These are not just theory concepts — every production NLP system, from Google Search to ChatGPT, starts with these exact steps.

---

### What You Will Learn Today

- Text Preprocessing — why cleaning raw text is the first and most important step
- Tokenization — the four types and when to use each
- Stopwords — when removing them helps and when it hurts
- Stemming vs Lemmatization — and how to choose between them
- Bag of Words and TF-IDF — how machines turn text into numbers

### Goal by End of Day

Build a complete mini sentiment classifier pipeline using everything covered today.

---

## Step 1 — Text Cleaning

### Why It Matters

Before any model can learn from text, the text needs to be clean. Raw text collected from the real world is messy. It contains things the model should not have to deal with:

- URLs (https://example.com)
- HTML tags (`<br>`, `<div>`, etc.)
- Emojis and special characters
- Punctuation
- Numbers (in many cases)
- Extra whitespace and newlines
- Mixed casing ("NLP" vs "nlp" vs "Nlp")

Text cleaning improves the **signal-to-noise ratio** — it helps the model focus on meaning rather than formatting artifacts.

### The Important Trade-off

Cleaning too aggressively can actually remove meaning. This is a judgment call every NLP engineer has to make.

A good example: if you are building a sentiment analysis model and you remove all emojis, you lose one of the strongest emotional signals in modern text. A review that says *"worst product ever"* carries a different emotional weight than *"worst product ever"* followed by a laughing emoji.

**Rule of thumb:** Clean what adds noise, preserve what carries meaning.

In [1]:
import re
import string

def clean_text(text, remove_emojis=True):
    """
    Basic text cleaning pipeline.
    Set remove_emojis=False for sentiment tasks where emojis carry signal.
    """
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove emojis (optional — depends on your task)
    if remove_emojis:
        text = re.sub(r'[^\x00-\x7F]+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Test it
raw = "Check this out!! https://example.com  Amazing product <br> 100% recommended"

print("Original : ", raw)
print("Cleaned  : ", clean_text(raw))

Original :  Check this out!! https://example.com  Amazing product <br> 100% recommended
Cleaned  :  check this out amazing product 100 recommended


---

## Step 2 — Tokenization

### What It Is

Tokenization is the process of splitting text into smaller units called **tokens**. It is always the first step after cleaning. The way you tokenize text has downstream consequences for everything else in your pipeline.

Tokenization affects:

- **Vocabulary size** — how many unique tokens your model needs to know
- **Model memory** — larger vocabularies require larger embedding tables
- **OOV problem** — Out-of-Vocabulary words that the model has never seen
- **Embedding mapping** — every token needs a vector representation

### The Four Types of Tokenization

**Word Tokenization**
Splits text on spaces and punctuation into individual words. Simple and intuitive. Works well for classical machine learning approaches. Problem: every new word not seen in training is OOV.

**Sentence Tokenization**
Splits a document into individual sentences. Used in tasks like summarization, question answering, and document understanding where sentence boundaries matter.

**Character Tokenization**
Splits text into individual characters. Produces very long sequences but handles noisy, informal, or misspelled text well. Rare in production.

**Subword Tokenization**
The modern standard. Splits words into meaningful subword units — common words stay whole, rare words are split into parts. This is what BERT, GPT, and all modern LLMs use.

Why subword? It solves the OOV problem without exploding vocabulary size. The word *"unbelievable"* might be split into *["un", "believe", "able"]* — each part has meaning and has been seen in training.

> **Key insight:** Subword tokenization is a careful balance between vocabulary size and semantic coverage. It is one of the reasons modern LLMs can handle any word, including made-up ones.

In [2]:
# Install required libraries
# !pip install nltk transformers -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize

text = "NLP is fascinating. Machines can now understand human language better than ever before."

# Word tokenization
word_tokens = word_tokenize(text)
print("Word Tokens:")
print(word_tokens)
print(f"Count: {len(word_tokens)}")

print()

# Sentence tokenization
sent_tokens = sent_tokenize(text)
print("Sentence Tokens:")
for i, s in enumerate(sent_tokens):
    print(f"  Sentence {i+1}: {s}")

print()

# Character tokenization
char_tokens = list(text[:20])  # first 20 chars for readability
print("Character Tokens (first 20 chars):")
print(char_tokens)

Word Tokens:
['NLP', 'is', 'fascinating', '.', 'Machines', 'can', 'now', 'understand', 'human', 'language', 'better', 'than', 'ever', 'before', '.']
Count: 15

Sentence Tokens:
  Sentence 1: NLP is fascinating.
  Sentence 2: Machines can now understand human language better than ever before.

Character Tokens (first 20 chars):
['N', 'L', 'P', ' ', 'i', 's', ' ', 'f', 'a', 's', 'c', 'i', 'n', 'a', 't', 'i', 'n', 'g', '.', ' ']


In [4]:
# Subword tokenization — how modern LLMs see text
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

examples = [
    "unbelievable",
    "tokenization",
    "supercalifragilistic",
    "AI is transforming the world"
]

print("Subword Tokenization (BERT)")
print("=" * 45)
for word in examples:
    tokens = tokenizer.tokenize(word)
    print(f"  {word:<30} -> {tokens}")

Subword Tokenization (BERT)
  unbelievable                   -> ['unbelievable']
  tokenization                   -> ['token', '##ization']
  supercalifragilistic           -> ['super', '##cal', '##if', '##rag', '##ilis', '##tic']
  AI is transforming the world   -> ['ai', 'is', 'transforming', 'the', 'world']


---

## Step 3 — Stopwords Removal

### What Stopwords Are

Stopwords are common words that appear frequently in nearly every document — words like *the, is, at, which, on, a, an*. Because they appear everywhere, they carry very little information about what makes one document different from another.

Removing them reduces the size of your vocabulary and the noise in your features.

### When Removal Helps

Stopword removal is most useful in **topic modeling** and **document classification** tasks where you want to identify the main subject of a piece of text. In these cases, high-frequency words dilute the signal from meaningful keywords.

### When Removal Hurts

Stopwords sometimes carry critical meaning. Consider these examples:

- *"The movie was good"* vs *"The movie was not good"* — removing *not* completely flips the sentiment
- *"Is this free?"* — removing *is* and *this* might lose the question structure
- *"Not bad at all"* — removing *not* turns a positive phrase into a negative one

> **Practical rule:** Remove stopwords in topic modeling and keyword extraction. Be cautious with removal in sentiment analysis and any task involving negation, questions, or emphasis.

In [5]:
import nltk
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

# Demonstrate the danger of removing stopwords in sentiment tasks
sentences = [
    "The movie was not good at all",
    "This is absolutely not what I expected",
    "I would not recommend this product"
]

print("Stopword Removal — Showing the Negation Risk")
print("=" * 55)

for sentence in sentences:
    tokens = word_tokenize(sentence.lower())
    filtered = [t for t in tokens if t not in stop_words]
    print(f"\nOriginal : {sentence}")
    print(f"Filtered : {' '.join(filtered)}")
    print(f"Risk     : 'not' removed — meaning changed")

Stopword Removal — Showing the Negation Risk

Original : The movie was not good at all
Filtered : movie good
Risk     : 'not' removed — meaning changed

Original : This is absolutely not what I expected
Filtered : absolutely expected
Risk     : 'not' removed — meaning changed

Original : I would not recommend this product
Filtered : would recommend product
Risk     : 'not' removed — meaning changed


---

## Step 4 — Stemming vs Lemmatization

### The Problem They Both Solve

Words appear in many forms: *run, runs, running, ran*. These are the same root concept, but a model treating them as separate tokens wastes vocabulary space and misses the connection between them.

Both stemming and lemmatization reduce words to a base form, but they do it in very different ways.

### Stemming

Stemming uses simple **rule-based patterns** to chop off word endings. It is fast and computationally cheap, but it is not linguistically aware. It can produce results that are not real words.

Examples of what stemming does:
- *running* -> *run*
- *happily* -> *happili* (not a real word)
- *studies* -> *studi* (not a real word)

### Lemmatization

Lemmatization uses a **dictionary and grammar rules** to find the true root form of a word, called the lemma. It is slower but always produces a valid word.

Examples of what lemmatization does:
- *running* -> *run*
- *happily* -> *happy*
- *studies* -> *study*
- *better* -> *good* (understands comparative forms)

### How to Choose

| Situation | Recommended Approach |
|---|---|
| Large-scale pipeline, speed is critical | Stemming |
| Semantic correctness matters | Lemmatization |
| Search indexing | Stemming |
| Question answering, information extraction | Lemmatization |
| Feeding into a transformer model | Neither (transformers handle this internally) |

> **Note:** If you are using a modern transformer model like BERT or GPT, you generally do not need stemming or lemmatization at all. These models learn word forms from context during training.

In [6]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["running", "studies", "happily", "better", "caring", "wolves", "feet", "goes"]

print(f"{'Word':<15} {'Stemmed':<15} {'Lemmatized':<15}")
print("-" * 45)
for word in words:
    stemmed    = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word, pos='v')  # pos='v' treats as verb
    print(f"{word:<15} {stemmed:<15} {lemmatized:<15}")

print()
print("Observation: Stemming sometimes produces non-words (e.g., 'happili').")
print("Lemmatization always returns a valid dictionary word.")

Word            Stemmed         Lemmatized     
---------------------------------------------
running         run             run            
studies         studi           study          
happily         happili         happily        
better          better          better         
caring          care            care           
wolves          wolv            wolves         
feet            feet            feet           
goes            goe             go             

Observation: Stemming sometimes produces non-words (e.g., 'happili').
Lemmatization always returns a valid dictionary word.


---

## Step 5 — Text Representation: Bag of Words and TF-IDF

After cleaning and preprocessing text, we still have a problem: **machine learning models require numbers, not words**. Text representation is how we convert text into numerical form that a model can actually learn from.

### Bag of Words (BoW)

Bag of Words represents a document as a count of how many times each word appears. Every document becomes a vector where each position corresponds to a word in the vocabulary, and the value is its frequency.

**What BoW ignores:**
- Word order — *"dog bites man"* and *"man bites dog"* produce the same vector
- Context — the meaning of a word in relation to surrounding words
- Semantics — *"great"* and *"excellent"* are treated as completely unrelated

**Why BoW still works:**
Despite its limitations, word frequency correlates strongly with topic. A review containing many occurrences of *"excellent", "love", "amazing"* is probably positive regardless of order. BoW remains a solid baseline for text classification.

### TF-IDF (Term Frequency — Inverse Document Frequency)

TF-IDF improves on BoW by not treating all words equally. It rewards words that are frequent in a specific document but rare across the whole corpus — these are the most informative words.

The formula is:

```
TF-IDF(word, document) = TF(word, document) x IDF(word)

where:
  TF  = frequency of word in this document
  IDF = log(total documents / documents containing the word)
```

**Concrete example:**
- The word *"movie"* appears in every film review. Its IDF is very low — it tells us almost nothing useful.
- The word *"masterpiece"* appears in very few reviews. Its IDF is high — it is a strong signal that this review is very positive.

> **Key insight:** TF-IDF approximates the importance of a word without needing any labeled data or supervised learning. It is one of the most elegant ideas in classical NLP.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

corpus = [
    "the movie was absolutely fantastic and brilliant",
    "the movie was terrible and boring",
    "brilliant performances made this movie a masterpiece",
    "the worst movie I have ever seen"
]

# Bag of Words
bow_vectorizer = CountVectorizer()
bow_matrix     = bow_vectorizer.fit_transform(corpus)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)
print("Bag of Words Matrix")
print(bow_df.to_string())

print()

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix     = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out()
)
print("TF-IDF Matrix")
print(tfidf_df.to_string())

print()
print("Observation: 'the' and 'movie' appear in all docs — TF-IDF gives them low scores.")
print("'masterpiece' appears once — TF-IDF gives it a high score.")

Bag of Words Matrix
   absolutely  and  boring  brilliant  ever  fantastic  have  made  masterpiece  movie  performances  seen  terrible  the  this  was  worst
0           1    1       0          1     0          1     0     0            0      1             0     0         0    1     0    1      0
1           0    1       1          0     0          0     0     0            0      1             0     0         1    1     0    1      0
2           0    0       0          1     0          0     0     1            1      1             1     0         0    0     1    0      0
3           0    0       0          0     1          0     1     0            0      1             0     1         0    1     0    0      1

TF-IDF Matrix
   absolutely    and  boring  brilliant   ever  fantastic   have   made  masterpiece  movie  performances   seen  terrible    the   this    was  worst
0       0.469  0.370   0.000      0.370  0.000      0.469  0.000  0.000        0.000  0.245         0.000  0.000  

---

## Day 2 Summary

Here is everything covered today and the key decision each concept requires in practice:

| Concept | What It Does | Key Decision |
|---|---|---|
| **Text Cleaning** | Removes noise from raw text | Decide what counts as noise vs. meaningful signal |
| **Word Tokenization** | Splits text into words | Best for classical ML |
| **Subword Tokenization** | Splits into meaningful subword units | Best for transformers |
| **Stopword Removal** | Removes common low-information words | Skip for sentiment — keep negations |
| **Stemming** | Fast rule-based root extraction | Use when speed matters more than accuracy |
| **Lemmatization** | Grammar-aware root extraction | Use when meaning matters more than speed |
| **Bag of Words** | Word frequency vectors | Simple, effective baseline |
| **TF-IDF** | Frequency weighted by rarity | Better than BoW for most tasks |


```

Tomorrow we move from simple word counts to real vector meaning — where *"king"* and *"queen"* are actually close in space.

---

*30 Days of AI — Day 2 of 30 | NLP Foundational Concepts*